In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers

@tf.keras.utils.register_keras_serializable()
class AnomalyDetectorNCE(tf.keras.Model):
    def __init__(self, latent_dim=8, **kwargs):
        super().__init__(**kwargs)

        # Encoder
        self.encoder = tf.keras.Sequential([
            layers.Dense(32, activation=None),
            layers.BatchNormalization(),
            layers.LeakyReLU(),
            layers.Dropout(0.1),
            layers.Dense(16, activation=None),
            layers.BatchNormalization(),
            layers.LeakyReLU(),
            layers.Dense(latent_dim, activation=None),
            layers.BatchNormalization(),
            layers.LeakyReLU(),
        ])

        # Decoder
        self.decoder = tf.keras.Sequential([
            layers.Dense(16, activation=None),
            layers.BatchNormalization(),
            layers.LeakyReLU(),
            layers.Dropout(0.1),
            layers.Dense(32, activation=None),
            layers.BatchNormalization(),
            layers.LeakyReLU(),
            layers.Dense(24)  # output dim
        ])

        # NCE Classifier Head
        self.classifier = tf.keras.Sequential([
            layers.Dense(8, activation='relu'),
            layers.Dense(1, activation='sigmoid')  # p(normal)
        ])

    def call(self, x):
        z = self.encoder(x)
        reconstruction = self.decoder(z)
        classification = self.classifier(z)
        return reconstruction, classification

    def get_config(self):
        base_config = super().get_config()
        return base_config

    @classmethod
    def from_config(cls, config):
        return cls(**config)


# Noise generation function for NCE
def generate_nce_noise(X, noise_ratio=1.0, perturb_std=0.5):
    """
    Generate negative samples for NCE training.
    Combines feature shuffling and Gaussian perturbation.
    
    Args:
        X: np.ndarray of normal samples
        noise_ratio: multiplier for how many noise samples to generate vs normal
        perturb_std: scaling factor for Gaussian noise
    Returns:
        np.ndarray of synthetic anomalies
    """
    n_samples = int(len(X) * noise_ratio)
    noise = np.copy(X[np.random.choice(len(X), n_samples, replace=True)])

    # Step 1: Feature shuffle (destroys correlations)
    for col in range(noise.shape[1]):
        np.random.shuffle(noise[:, col])

    # Step 2: Gaussian perturbation
    col_std = np.std(X, axis=0)
    noise += np.random.normal(0, perturb_std * col_std, size=noise.shape)

    return noise


C:\Users\Rohan\miniconda3\envs\qae\Lib\site-packages\google\protobuf\runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/attr_value.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
C:\Users\Rohan\miniconda3\envs\qae\Lib\site-packages\google\protobuf\runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/tensor.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
C:\Users\Rohan\miniconda3\envs\qae\Lib\site-packages\google\protobuf\runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/resource_handle.proto. Please update the gencode to avoid

In [3]:
import pandas as pd

transformed_training_data = pd.read_csv('Dataset/Transformed/Normalised/labelled_transformed_training_data.csv')

# 1. Extract labels (column 6)
training_labels = transformed_training_data.iloc[:, 6].astype(bool)  # Anomalies: True

# 2. Drop label and extra column (column 6 and 7)
training_data = transformed_training_data.drop(
    columns=[transformed_training_data.columns[6], transformed_training_data.columns[7]]
)

# 3. Convert to float32 tensor
training_data = tf.convert_to_tensor(training_data.values, dtype=tf.float32)

# 4. Convert labels to tensor (bool)
training_labels = tf.convert_to_tensor(training_labels.values, dtype=tf.bool)

# 5. Split normal vs anomalous
normal_training_data = tf.boolean_mask(training_data, ~training_labels)
anomalous_training_data = tf.boolean_mask(training_data, training_labels)

transformed_validation_data = pd.read_csv('Dataset/Transformed/Normalised/labelled_transformed_validation_data.csv')

# 1. Extract labels (column 6)
validation_labels = transformed_validation_data.iloc[:, 6].astype(bool)  # Anomalies: True

# 2. Drop label and extra column (column 6 and 7)
validation_data = transformed_validation_data.drop(
    columns=[transformed_validation_data.columns[6], transformed_validation_data.columns[7]]
)

# 3. Convert to float32 tensor
validation_data = tf.convert_to_tensor(validation_data.values, dtype=tf.float32)

# 4. Convert labels to tensor (bool)
validation_labels = tf.convert_to_tensor(validation_labels.values, dtype=tf.bool)

# 5. Split normal vs anomalous
normal_validation_data = tf.boolean_mask(validation_data, ~validation_labels)
anomalous_validation_data = tf.boolean_mask(validation_data, validation_labels)

transformed_testing_data = pd.read_csv('Dataset/Transformed/Normalised/labelled_transformed_testing_data.csv')

# 1. Extract labels (column 6)
testing_labels = transformed_testing_data.iloc[:, 6].astype(bool)  # Anomalies: True

# 2. Drop label and extra column (column 6 and 7)
testing_data = transformed_testing_data.drop(
    columns=[transformed_testing_data.columns[6], transformed_testing_data.columns[7]]
)

# 3. Convert to float32 tensor
testing_data = tf.convert_to_tensor(testing_data.values, dtype=tf.float32)

# 4. Convert labels to tensor (bool)
testing_labels = tf.convert_to_tensor(testing_labels.values, dtype=tf.bool)

# 5. Split normal vs anomalous
normal_testing_data = tf.boolean_mask(testing_data, ~testing_labels)
anomalous_testing_data = tf.boolean_mask(testing_data, testing_labels)

In [5]:
# Example training pipeline
X_normal = normal_training_data.numpy()
X_noise = generate_nce_noise(X_normal, noise_ratio=1.0, perturb_std=0.5)

X_all = np.vstack([X_normal, X_noise])
y_all = np.hstack([np.ones(len(X_normal)), np.zeros(len(X_noise))])

X_val_normal = normal_validation_data.numpy()
X_val_noise = generate_nce_noise(X_val_normal, noise_ratio=1.0, perturb_std=0.5)
X_val_all = np.vstack([X_val_normal, X_val_noise])
y_val_all = np.hstack([np.ones(len(X_val_normal)), np.zeros(len(X_val_noise))])

model = AnomalyDetectorNCE()

model.compile(
    optimizer='adam',
    loss=['mse', 'binary_crossentropy'],
    loss_weights=[1.0, 0.5]  # reconstruction vs classification
)

early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5,
    min_lr=1e-6
)

model.fit(
    X_all, [X_all, y_all],
    batch_size=64,
    epochs=20,
    validation_data=(X_val_all, [X_val_all, y_val_all]),
    callbacks=[early_stopping, reduce_lr],
    shuffle=True
)


Epoch 1/20
23809/23809 ━━━━━━━━━━━━━━━━━━━━ 51s 2ms/step - binary_crossentropy_loss: 0.1184 - loss: 0.0613 - mse_loss: 0.0021 - val_binary_crossentropy_loss: 2.4687 - val_loss: 1.2360 - val_mse_loss: 0.0016 - learning_rate: 0.0010
Epoch 2/20
23809/23809 ━━━━━━━━━━━━━━━━━━━━ 47s 2ms/step - binary_crossentropy_loss: 0.0489 - loss: 0.0256 - mse_loss: 0.0011 - val_binary_crossentropy_loss: 3.2134 - val_loss: 1.6077 - val_mse_loss: 9.2395e-04 - learning_rate: 0.0010
Epoch 3/20
23809/23809 ━━━━━━━━━━━━━━━━━━━━ 48s 2ms/step - binary_crossentropy_loss: 0.0366 - loss: 0.0189 - mse_loss: 6.2837e-04 - val_binary_crossentropy_loss: 3.5205 - val_loss: 1.7610 - val_mse_loss: 6.7041e-04 - learning_rate: 0.0010
Epoch 4/20
23809/23809 ━━━━━━━━━━━━━━━━━━━━ 50s 2ms/step - binary_crossentropy_loss: 0.0297 - loss: 0.0153 - mse_loss: 4.9011e-04 - val_binary_crossentropy_loss: 2.2110 - val_loss: 1.1062 - val_mse_loss: 5.9145e-04 - learning_rate: 0.0010
Epoch 5/20
23809/23809 ━━━━━━━━━━━━━━━━━━━━ 50s 2ms/step

In [6]:
model.save('Models/CAE/8f_cae_nce_model.keras')